# Notebook 2 — Préparation, standardisation et contrôle qualité

## Position dans le pipeline

`NB1 → **NB2 (vous êtes ici)** → NB3 → NB4`  
Guide : `00_PIPELINE_PAS_A_PAS.md`

## Objectif

Transformer le dataset brut en **preuves de qualité** (drapeaux + métriques), sans certification métrologique.

## Entrées attendues (Notebook 1)

- `data/raw/opensensemap_raw_long_active_50.csv`
- `data/metadata/stations_selectionnees_actives.csv`

## Traitements réalisés

1. audit initial et traçabilité des rejets ;
2. harmonisation schéma / unités (vectorisée) ;
3. doublons, fréquence, complétude, gaps ;
4. plausibilité physique, valeurs figées, sauts, anomalies statistiques ;
5. contrôle spatial facultatif ;
6. export pour le Notebook 3.

## Sorties attendues

- `data/processed/opensensemap_measurements_qc_flagged.csv` (+ parquet)
- `data/processed/opensensemap_qc_metrics_station_variable.csv`
- `data/processed/opensensemap_rejected_rows.csv`
- `data/processed/opensensemap_duplicates_removed.csv`
- `data/processed/opensensemap_qc_parameters.json`

> Une anomalie détectée n’est pas automatiquement une erreur. La valeur originale est conservée ; la règle est documentée.

Le QC série par série (figé / sauts / MAD) reprend la logique de `implement_trust_framework.py` (boucle unique, adaptée aux ~2,4 M lignes).

**Alternative CLI (QC + confiance NB3) :**

```bash
python implement_trust_framework.py
```


## 1. Bibliothèques et configuration générale

In [ ]:
from pathlib import Path
from IPython.display import display
import json
import math
import re
import unicodedata
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

RANDOM_STATE = 42

In [ ]:
# Arborescence alignée sur le Notebook 1 (data/ sous le dossier des notebooks)
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
METADATA_DIR = DATA_DIR / "metadata"
REPORTS_DIR = PROJECT_ROOT / "reports"

for folder in [DATA_DIR, RAW_DIR, PROCESSED_DIR, METADATA_DIR, REPORTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Chemins par défaut = sorties du Notebook 1 (surchargeables si besoin)
MEASUREMENTS_FILE = "data/raw/opensensemap_raw_long_active_50.csv"
STATIONS_FILE = "data/metadata/stations_selectionnees_actives.csv"

MEASUREMENT_CANDIDATES = [
    "opensensemap_raw_long_active_50.csv",
    "opensensemap_measurements_raw.csv",
    "measurements_raw.csv",
    "observations_raw.csv",
    "opensensemap_measurements.csv",
    "dataset_opensensemap.csv",
]

STATION_CANDIDATES = [
    "stations_selectionnees_actives.csv",
    "stations_selectionnees.csv",
    "opensensemap_stations.csv",
    "stations_metadata.csv",
    "stations.csv",
    "senseboxes.csv",
]

TARGET_VARIABLES = ["temperature", "humidity", "pressure"]

# Paramètres de l'étude : à documenter et à adapter au contexte.
PLAUSIBILITY_LIMITS = {
    "temperature": (-30.0, 50.0),   # °C
    "humidity": (0.0, 100.0),       # %
    "pressure": (850.0, 1100.0),    # hPa
}

CANONICAL_UNITS = {
    "temperature": "°C",
    "humidity": "%",
    "pressure": "hPa",
}

GAP_FACTOR = 3.0

STUCK_MIN_DURATION_HOURS = 2.0
STUCK_MIN_OBSERVATIONS = 4
ROUNDING_DIGITS = {
    "temperature": 1,
    "humidity": 1,
    "pressure": 1,
}

JUMP_MAD_FACTOR = 6.0
MIN_JUMP_RATE_PER_HOUR = {
    "temperature": 8.0,
    "humidity": 30.0,
    "pressure": 5.0,
}

ROBUST_Z_THRESHOLD = 4.5
MIN_GROUP_SIZE_FOR_STAT_QC = 20

# Les conversions reposent par défaut sur l'unité déclarée.
ALLOW_UNIT_INFERENCE = False

# Contrôle spatial désactivé par défaut.
RUN_SPATIAL_QC = False
SPATIAL_RADIUS_KM = 30.0
SPATIAL_MIN_NEIGHBORS = 2
SPATIAL_ROBUST_Z_THRESHOLD = 4.5
SPATIAL_ABS_FLOOR = {
    "temperature": 3.0,
    "humidity": 15.0,
    "pressure": 5.0,
}

# Sorties attendues (contrôle rapide)
QC_FLAGGED_CSV = PROCESSED_DIR / "opensensemap_measurements_qc_flagged.csv"
QC_METRICS_CSV = PROCESSED_DIR / "opensensemap_qc_metrics_station_variable.csv"
QC_REJECTED_CSV = PROCESSED_DIR / "opensensemap_rejected_rows.csv"
QC_DUPLICATES_CSV = PROCESSED_DIR / "opensensemap_duplicates_removed.csv"
QC_PARAMS_JSON = PROCESSED_DIR / "opensensemap_qc_parameters.json"

print("Projet      :", PROJECT_ROOT)
print("Entrées raw :", RAW_DIR)
print("Métadonnées :", METADATA_DIR)
print("Sorties     :", PROCESSED_DIR)
print("Mesures     :", MEASUREMENTS_FILE)
print("Stations    :", STATIONS_FILE)


## 0. Contrôle rapide — sorties QC déjà disponibles ?

Si les fichiers `data/processed/opensensemap_*qc*` existent déjà (run antérieur ou `implement_trust_framework.py`), vous pouvez **sauter** le QC et passer au **Notebook 3**.

Sinon, exécuter les cellules suivantes jusqu’à l’export.


In [ ]:
expected_qc = [
    QC_FLAGGED_CSV,
    QC_METRICS_CSV,
    QC_REJECTED_CSV,
    QC_DUPLICATES_CSV,
    QC_PARAMS_JSON,
]
present = [p for p in expected_qc if p.exists()]
missing = [p for p in expected_qc if not p.exists()]

print(f"Sorties QC présentes : {len(present)}/{len(expected_qc)}")
for p in present:
    print("  OK ", p.name)
for p in missing:
    print("  -- ", p.name)

if not missing:
    print("\nQC déjà disponible → vous pouvez ouvrir le Notebook 3.")
else:
    print("\nExécuter le Notebook 2 jusqu'à l'export (ou : python implement_trust_framework.py).")


## 2. Fonctions utilitaires

In [5]:
def normalize_text(value):
    """Normalise un texte pour faciliter les comparaisons."""
    if pd.isna(value):
        return ""
    text = str(value).strip().lower()
    text = "".join(
        char for char in unicodedata.normalize("NFKD", text)
        if not unicodedata.combining(char)
    )
    text = re.sub(r"[^a-z0-9%°/_-]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def normalize_column_name(name):
    text = normalize_text(name)
    return re.sub(r"[\s/-]+", "_", text)


COLUMN_ALIASES = {
    "station_id": [
        "station_id", "box_id", "sensebox_id", "senseboxid",
        "boxid", "stationid", "_id"
    ],
    "station_name": [
        "station_name", "box_name", "sensebox_name",
        "station", "box", "name"
    ],
    "sensor_id": [
        "sensor_id", "sensorid", "sensor", "sensor_id_open"
    ],
    "variable": [
        "variable", "phenomenon", "sensor_title", "sensor_name",
        "title", "measurement_type", "type"
    ],
    "unit": [
        "unit", "sensor_unit", "measurement_unit", "unite"
    ],
    "timestamp": [
        "timestamp", "createdat", "created_at", "time",
        "datetime", "date", "measured_at"
    ],
    "value": [
        "value", "measurement", "measurement_value",
        "valeur", "reading"
    ],
    "latitude": ["latitude", "lat", "y"],
    "longitude": ["longitude", "lon", "lng", "long", "x"],
    "exposure": ["exposure", "exposition"],
    "model": ["model", "modele", "sensor_model"],
}


def harmonize_column_names(df):
    """Renomme les colonnes reconnues vers un schéma canonique."""
    result = df.copy()
    normalized_to_original = {
        normalize_column_name(col): col for col in result.columns
    }

    rename_map = {}
    for canonical, aliases in COLUMN_ALIASES.items():
        for alias in aliases:
            alias_norm = normalize_column_name(alias)
            if alias_norm in normalized_to_original:
                original = normalized_to_original[alias_norm]
                if canonical not in result.columns:
                    rename_map[original] = canonical
                break

    return result.rename(columns=rename_map)


def resolve_file(explicit_path, directory, candidates, required=True):
    """Résout un chemin explicite ou recherche un fichier candidat."""
    if explicit_path:
        path = Path(explicit_path)
        if not path.is_absolute():
            path = PROJECT_ROOT / path
        if path.exists():
            return path
        raise FileNotFoundError(f"Fichier introuvable : {path}")

    for filename in candidates:
        candidate = directory / filename
        if candidate.exists():
            return candidate

    available = (
        sorted(directory.glob("*.csv"))
        + sorted(directory.glob("*.parquet"))
        + sorted(directory.glob("*.xlsx"))
        + sorted(directory.glob("*.xls"))
    )

    if len(available) == 1:
        return available[0]

    if required:
        expected = "\n- ".join(candidates)
        raise FileNotFoundError(
            f"Aucun fichier d'entrée reconnu dans {directory}.\n"
            f"Noms attendus :\n- {expected}\n"
            "Renseignez MEASUREMENTS_FILE dans la cellule de configuration."
        )
    return None


def read_table(path):
    """Lit un fichier CSV, Parquet ou Excel."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path, low_memory=False)
    if suffix == ".parquet":
        return pd.read_parquet(path)
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    raise ValueError(f"Format non pris en charge : {suffix}")


def canonical_variable(value):
    """Ramène les noms de phénomènes vers trois variables canoniques."""
    text = normalize_text(value)

    temperature_terms = [
        "temperature", "temperatur", "temp", "lufttemperatur",
        "air temperature", "air_temperature"
    ]
    humidity_terms = [
        "humidity", "humidite", "luftfeuchte", "relative humidity",
        "relative_humidity", "rel humidity", "feuchte"
    ]
    pressure_terms = [
        "pressure", "pression", "luftdruck", "barometric pressure",
        "barometer", "atmospheric pressure"
    ]

    if any(term in text for term in temperature_terms):
        return "temperature"
    if any(term in text for term in humidity_terms):
        return "humidity"
    if any(term in text for term in pressure_terms):
        return "pressure"
    return "other"


def safe_rate(numerator, denominator):
    if denominator in (0, None) or pd.isna(denominator):
        return np.nan
    return numerator / denominator

## 3. Chargement des fichiers produits par le Notebook 1

In [6]:
measurements_path = resolve_file(
    MEASUREMENTS_FILE,
    RAW_DIR,
    MEASUREMENT_CANDIDATES,
    required=True,
)

# Métadonnées : metadata/ en priorité, puis raw/
stations_path = resolve_file(
    STATIONS_FILE,
    METADATA_DIR,
    STATION_CANDIDATES,
    required=False,
)
if stations_path is None:
    stations_path = resolve_file(
        STATIONS_FILE,
        RAW_DIR,
        STATION_CANDIDATES,
        required=False,
    )

measurements_raw = read_table(measurements_path)
stations_raw = read_table(stations_path) if stations_path else pd.DataFrame()

print(f"Mesures chargées : {measurements_path}")
print(f"Dimensions       : {measurements_raw.shape}")

if stations_path:
    print(f"Métadonnées      : {stations_path}")
    print(f"Dimensions       : {stations_raw.shape}")
else:
    print("Aucun fichier séparé de métadonnées n'a été détecté.")

display(measurements_raw.head())

Mesures chargées : d:\SI\notebook\notebook\data\raw\opensensemap_raw_long_active_50.csv
Dimensions       : (2448207, 14)
Métadonnées      : d:\SI\notebook\notebook\data\metadata\stations_selectionnees_actives.csv
Dimensions       : (38, 29)


,station_id,station_name,latitude,longitude,exposure,phenomenon,sensor_id,sensor_title,sensor_unit,sensor_type,timestamp,value,chunk_start,chunk_end
0,578207d56fea661300861f3b,emp-wetter,51.6603,9.3913,outdoor,humidity,578207d56fea661300861f40,rel. Luftfeuchte,%,HDC1008,2026-06-15 00:01:13.443000+00:00,81.7200,2026-06-15T00:00:00Z,2026-06-22T00:00:00Z
1,578207d56fea661300861f3b,emp-wetter,51.6603,9.3913,outdoor,humidity,578207d56fea661300861f40,rel. Luftfeuchte,%,HDC1008,2026-06-15 00:02:43.170000+00:00,81.7600,2026-06-15T00:00:00Z,2026-06-22T00:00:00Z
2,578207d56fea661300861f3b,emp-wetter,51.6603,9.3913,outdoor,humidity,578207d56fea661300861f40,rel. Luftfeuchte,%,HDC1008,2026-06-15 00:04:17.770000+00:00,81.9100,2026-06-15T00:00:00Z,2026-06-22T00:00:00Z
3,578207d56fea661300861f3b,emp-wetter,51.6603,9.3913,outdoor,humidity,578207d56fea661300861f40,rel. Luftfeuchte,%,HDC1008,2026-06-15 00:05:56.838000+00:00,81.9000,2026-06-15T00:00:00Z,2026-06-22T00:00:00Z
4,578207d56fea661300861f3b,emp-wetter,51.6603,9.3913,outdoor,humidity,578207d56fea661300861f40,rel. Luftfeuchte,%,HDC1008,2026-06-15 00:07:34.050000+00:00,82.1100,2026-06-15T00:00:00Z,2026-06-22T00:00:00Z


## 4. Harmonisation du schéma

In [ ]:
measurements = harmonize_column_names(measurements_raw)
stations = harmonize_column_names(stations_raw) if not stations_raw.empty else pd.DataFrame()

required_columns = ["station_id", "variable", "timestamp", "value"]
missing_required = [col for col in required_columns if col not in measurements.columns]

if missing_required:
    raise KeyError(
        "Colonnes obligatoires absentes après harmonisation : "
        + ", ".join(missing_required)
        + "\nColonnes disponibles : "
        + ", ".join(map(str, measurements.columns))
    )

for col in ["station_name", "sensor_id", "unit", "latitude", "longitude"]:
    if col not in measurements.columns:
        measurements[col] = np.nan

# Fusion prudente avec les métadonnées de station.
if not stations.empty and "station_id" in stations.columns:
    metadata_cols = [
        col for col in [
            "station_id", "station_name", "latitude", "longitude",
            "exposure", "model"
        ]
        if col in stations.columns
    ]
    metadata = stations[metadata_cols].drop_duplicates(subset=["station_id"])

    measurements = measurements.merge(
        metadata,
        on="station_id",
        how="left",
        suffixes=("", "_meta"),
    )

    for col in ["station_name", "latitude", "longitude"]:
        meta_col = f"{col}_meta"
        if meta_col in measurements.columns:
            measurements[col] = measurements[col].combine_first(measurements[meta_col])
            measurements.drop(columns=meta_col, inplace=True)

print("Colonnes harmonisées :")
print(list(measurements.columns))
display(measurements.head())

## 5. Audit initial et traçabilité des lignes rejetées

In [ ]:
data = measurements.copy()
data["row_id_raw"] = np.arange(len(data))

data["value_original"] = data["value"]
data["unit_original"] = data["unit"]
data["variable_original"] = data["variable"]

data["station_id"] = data["station_id"].astype("string").str.strip()
data["sensor_id"] = data["sensor_id"].astype("string").str.strip()
data["timestamp"] = pd.to_datetime(data["timestamp"], errors="coerce", utc=True)
data["value"] = pd.to_numeric(data["value"], errors="coerce")
data["latitude"] = pd.to_numeric(data["latitude"], errors="coerce")
data["longitude"] = pd.to_numeric(data["longitude"], errors="coerce")
data["variable"] = data["variable"].map(canonical_variable)

conditions = [
    data["station_id"].isna() | data["station_id"].eq(""),
    data["timestamp"].isna(),
    data["value"].isna(),
    data["variable"].eq("other"),
]
reasons = [
    "station_id_manquant",
    "timestamp_invalide",
    "valeur_non_numerique",
    "variable_non_cible",
]

data["rejection_reason"] = ""
for condition, reason in zip(conditions, reasons):
    data.loc[
        condition & data["rejection_reason"].eq(""),
        "rejection_reason"
    ] = reason

rejected_rows = data[data["rejection_reason"].ne("")].copy()
qc = data[data["rejection_reason"].eq("")].copy()
qc = qc[qc["variable"].isin(TARGET_VARIABLES)].copy()

print(f"Lignes brutes                     : {len(data):,}")
print(f"Lignes rejetées ou hors périmètre : {len(rejected_rows):,}")
print(f"Lignes retenues pour le QC        : {len(qc):,}")

if not rejected_rows.empty:
    display(
        rejected_rows["rejection_reason"]
        .value_counts(dropna=False)
        .rename_axis("motif")
        .reset_index(name="nombre")
    )

## 6. Harmonisation prudente des unités

In [ ]:
# Harmonisation des unités — version vectorisée
# (équivalent de std_units() dans implement_trust_framework.py)

def standardize_units_vectorized(qc_df):
    unit = (
        qc_df["unit"].astype(str).str.lower()
        .str.replace(" ", "", regex=False)
    )
    # normaliser quelques variantes courantes
    unit = unit.str.replace("°", "°", regex=False)
    variable = qc_df["variable"]
    value = qc_df["value"].astype(float)
    value_std = value.copy()
    converted = pd.Series(False, index=qc_df.index)
    recognized = pd.Series(False, index=qc_df.index)

    m = variable.eq("temperature")
    temp_c = m & unit.isin({"°c", "c", "celsius", "degc", "degreecelsius", "c°"})
    temp_k = m & unit.isin({"k", "kelvin"})
    temp_f = m & unit.isin({"°f", "f", "fahrenheit", "degf"})
    recognized = recognized | temp_c | temp_k | temp_f
    value_std = value_std.mask(temp_k, value - 273.15)
    value_std = value_std.mask(temp_f, (value - 32.0) * 5.0 / 9.0)
    converted = converted | temp_k | temp_f

    m = variable.eq("humidity")
    hum_pct = m & (
        unit.isin({"%", "percent", "percentage", "prozent", "%rf", "rf", "rel%"})
        | unit.str.contains("%|percent|prozent|rf", regex=True, na=False)
    )
    hum_frac = m & unit.isin({"fraction", "ratio", "0-1", "01"})
    recognized = recognized | hum_pct | hum_frac
    value_std = value_std.mask(hum_frac, value * 100.0)
    converted = converted | hum_frac

    m = variable.eq("pressure")
    p_hpa = m & unit.isin({"hpa", "mbar", "millibar"})
    p_pa = m & unit.isin({"pa", "pascal", "pascals"})
    p_kpa = m & unit.isin({"kpa", "kilopascal", "kilopascals"})
    recognized = recognized | p_hpa | p_pa | p_kpa
    value_std = value_std.mask(p_pa, value / 100.0)
    value_std = value_std.mask(p_kpa, value * 10.0)
    converted = converted | p_pa | p_kpa

    out = qc_df.copy()
    out["value_std"] = value_std
    out["unit_std"] = variable.map(CANONICAL_UNITS)
    out["unit_conversion_applied"] = converted
    out["flag_unit_unknown"] = ~recognized
    return out


qc = standardize_units_vectorized(qc)

unit_summary = (
    qc.groupby(["variable", "unit_original"], dropna=False)
    .agg(
        nombre=("value_std", "size"),
        conversions=("unit_conversion_applied", "sum"),
        unites_non_reconnues=("flag_unit_unknown", "sum"),
    )
    .reset_index()
    .sort_values(["variable", "nombre"], ascending=[True, False])
)

display(unit_summary)
print("Unités standardisées (vectorisé).")


## 7. Détection et traitement des doublons

In [ ]:
# Clé de série : sensor_id si disponible, sinon station × variable
sensor_available = (
    qc["sensor_id"].notna()
    & qc["sensor_id"].ne("<NA>")
    & qc["sensor_id"].ne("")
)

qc["_series_key"] = np.where(
    sensor_available,
    qc["station_id"].astype(str) + "|" + qc["sensor_id"].astype(str),
    qc["station_id"].astype(str) + "|" + qc["variable"].astype(str),
)

# Aligné pipeline / script : station × variable × timestamp
# (équivalent pratique via _series_key × timestamp)
duplicate_keys = ["_series_key", "timestamp"]

qc["flag_duplicate_timestamp"] = qc.duplicated(
    subset=duplicate_keys,
    keep="first",
)

duplicates_all = qc[qc.duplicated(subset=duplicate_keys, keep=False)].copy()
duplicates_removed = qc[qc["flag_duplicate_timestamp"]].copy()

qc = qc.loc[~qc["flag_duplicate_timestamp"]].copy()
# Les lignes conservées ne sont plus marquées comme doublons (comme le script)
qc["flag_duplicate_timestamp"] = False

qc = qc.sort_values(
    ["station_id", "variable", "_series_key", "timestamp"]
).reset_index(drop=True)

print(f"Observations dupliquées détectées : {len(duplicates_all):,}")
print(f"Répétitions retirées              : {len(duplicates_removed):,}")
print(f"Lignes après dédoublonnage        : {len(qc):,}")


## 8. Estimation de la fréquence et analyse de la complétude

In [ ]:
SERIES_COLS = ["station_id", "variable", "_series_key"]

qc["delta_seconds"] = (
    qc.groupby(SERIES_COLS, dropna=False)["timestamp"]
    .diff()
    .dt.total_seconds()
)

positive_deltas = qc.loc[qc["delta_seconds"] > 0].copy()

frequency_metrics = (
    positive_deltas.groupby(SERIES_COLS, dropna=False)
    .agg(
        expected_interval_seconds=("delta_seconds", "median"),
        interval_mean_seconds=("delta_seconds", "mean"),
        interval_std_seconds=("delta_seconds", "std"),
        interval_q25_seconds=("delta_seconds", lambda s: s.quantile(0.25)),
        interval_q75_seconds=("delta_seconds", lambda s: s.quantile(0.75)),
    )
    .reset_index()
)

qc = qc.merge(
    frequency_metrics[SERIES_COLS + ["expected_interval_seconds"]],
    on=SERIES_COLS,
    how="left",
)


def summarize_series(group):
    start = group["timestamp"].min()
    end = group["timestamp"].max()
    observed_count = group["timestamp"].nunique()
    expected_intervals = group["expected_interval_seconds"].dropna()

    if expected_intervals.empty or start == end:
        expected_count = observed_count
    else:
        interval = expected_intervals.iloc[0]
        expected_count = int(
            math.floor((end - start).total_seconds() / interval) + 1
        )

    completeness = (
        min(1.0, safe_rate(observed_count, expected_count))
        if expected_count else np.nan
    )

    deltas = group["delta_seconds"].dropna()
    longest_gap_seconds = deltas.max() if not deltas.empty else np.nan

    return pd.Series({
        "start_time": start,
        "end_time": end,
        "observed_count": observed_count,
        "expected_count": expected_count,
        "estimated_missing_count": max(0, expected_count - observed_count),
        "completeness_rate": completeness,
        "longest_gap_hours": (
            longest_gap_seconds / 3600
            if pd.notna(longest_gap_seconds) else np.nan
        ),
    })


series_coverage = (
    qc.groupby(SERIES_COLS, dropna=False, group_keys=False)
    .apply(summarize_series)
    .reset_index()
)

series_coverage = series_coverage.merge(
    frequency_metrics,
    on=SERIES_COLS,
    how="left",
)

display(
    series_coverage
    .sort_values("completeness_rate")
    .head(20)
)

## 9. Contrôle de plausibilité physique

In [ ]:
qc["plausibility_min"] = qc["variable"].map(
    {variable: bounds[0] for variable, bounds in PLAUSIBILITY_LIMITS.items()}
)
qc["plausibility_max"] = qc["variable"].map(
    {variable: bounds[1] for variable, bounds in PLAUSIBILITY_LIMITS.items()}
)

qc["flag_physical_implausibility"] = (
    (qc["value_std"] < qc["plausibility_min"])
    | (qc["value_std"] > qc["plausibility_max"])
)

plausibility_summary = (
    qc.groupby("variable")
    .agg(
        observations=("value_std", "size"),
        anomalies_physiques=("flag_physical_implausibility", "sum"),
    )
    .assign(
        taux_anomalies_physiques=lambda d:
        d["anomalies_physiques"] / d["observations"]
    )
    .reset_index()
)

display(plausibility_summary)

## 10. Détection des interruptions temporelles

In [ ]:
qc["gap_threshold_seconds"] = (
    GAP_FACTOR * qc["expected_interval_seconds"]
)

qc["flag_gap_before"] = (
    qc["delta_seconds"].notna()
    & qc["gap_threshold_seconds"].notna()
    & (qc["delta_seconds"] > qc["gap_threshold_seconds"])
)

qc["gap_duration_hours"] = np.where(
    qc["flag_gap_before"],
    qc["delta_seconds"] / 3600.0,
    0.0,
)

gap_summary = (
    qc.groupby(SERIES_COLS, dropna=False)
    .agg(
        nombre_interruptions=("flag_gap_before", "sum"),
        plus_longue_interruption_h=("gap_duration_hours", "max"),
    )
    .reset_index()
    .sort_values("plus_longue_interruption_h", ascending=False)
)

display(gap_summary.head(20))

## 11–13. Contrôles par série (figé, sauts, anomalies)

Une seule passe par série `station × variable`, comme dans `implement_trust_framework.py` — bien plus rapide que des `.apply` successifs sur ~2 M lignes.


In [ ]:
# Une passe : valeurs figées + sauts temporels + z-score robuste
SERIES_QC_COLS = ["station_id", "variable", "_series_key"]

parts = []
n_series = qc.groupby(SERIES_QC_COLS, dropna=False).ngroups
print(f"Contrôle série par série sur {n_series} séries...")

for i, (_, group) in enumerate(
    qc.groupby(SERIES_QC_COLS, dropna=False, sort=False), start=1
):
    g = group.sort_values("timestamp").copy()
    variable = g["variable"].iloc[0]
    digits = ROUNDING_DIGITS.get(variable, 1)

    # --- valeurs figées ---
    rounded = g["value_std"].round(digits)
    run_id = rounded.ne(rounded.shift()).cumsum()
    duration_h = g.groupby(run_id)["timestamp"].transform(
        lambda s: (s.max() - s.min()).total_seconds() / 3600.0
    )
    run_counts = run_id.map(run_id.value_counts())
    g["flag_stuck_value"] = (
        (run_counts >= STUCK_MIN_OBSERVATIONS)
        & (duration_h >= STUCK_MIN_DURATION_HOURS)
    )

    # --- sauts temporels ---
    delta_seconds = g["delta_seconds"] if "delta_seconds" in g.columns else (
        g["timestamp"].diff().dt.total_seconds()
    )
    rate_h = g["value_std"].diff().abs() * 3600.0 / delta_seconds
    rate_h = rate_h.replace([np.inf, -np.inf], np.nan)
    rates = rate_h.dropna()
    thr = MIN_JUMP_RATE_PER_HOUR.get(variable, 0.0)
    if len(rates) >= 5:
        med = rates.median()
        mad = (rates - med).abs().median()
        if mad and not np.isnan(mad):
            thr = max(thr, float(med + JUMP_MAD_FACTOR * 1.4826 * mad))
    g["change_rate_per_hour"] = rate_h
    g["jump_threshold_per_hour"] = thr
    g["flag_temporal_jump"] = rate_h.gt(thr).fillna(False)

    # --- anomalies statistiques (MAD) ---
    values = g["value_std"]
    g["robust_z"] = np.nan
    g["flag_statistical_anomaly"] = False
    if values.notna().sum() >= MIN_GROUP_SIZE_FOR_STAT_QC:
        med = values.median()
        mad = (values - med).abs().median()
        if mad and not np.isnan(mad) and mad != 0:
            robust_z = 0.6745 * (values - med) / mad
            g["robust_z"] = robust_z
            g["flag_statistical_anomaly"] = robust_z.abs() > ROBUST_Z_THRESHOLD

    parts.append(g)
    if i % 20 == 0 or i == n_series:
        print(f"  {i}/{n_series} séries", flush=True)

qc = pd.concat(parts, ignore_index=True)
print("Contrôles par série terminés.")


## Résumé — valeurs figées


In [ ]:
stuck_summary = (
    qc.groupby(SERIES_QC_COLS, dropna=False)
    .agg(
        observations_figees=("flag_stuck_value", "sum"),
        observations=("value_std", "size"),
    )
    .assign(
        taux_valeurs_figees=lambda d:
        d["observations_figees"] / d["observations"]
    )
    .reset_index()
    .sort_values("taux_valeurs_figees", ascending=False)
)

display(stuck_summary.head(20))


## Résumé — sauts temporels


In [ ]:
jump_summary = (
    qc.groupby(SERIES_QC_COLS, dropna=False)
    .agg(
        sauts_detectes=("flag_temporal_jump", "sum"),
        observations=("value_std", "size"),
        seuil_saut_par_heure=("jump_threshold_per_hour", "first"),
    )
    .assign(
        taux_sauts=lambda d:
        d["sauts_detectes"] / d["observations"]
    )
    .reset_index()
    .sort_values("taux_sauts", ascending=False)
)

display(jump_summary.head(20))


## Résumé — anomalies statistiques (MAD)


In [ ]:
stat_summary = (
    qc.groupby(SERIES_QC_COLS, dropna=False)
    .agg(
        anomalies_statistiques=("flag_statistical_anomaly", "sum"),
        observations=("value_std", "size"),
    )
    .assign(
        taux_anomalies_statistiques=lambda d:
        d["anomalies_statistiques"] / d["observations"]
    )
    .reset_index()
    .sort_values("taux_anomalies_statistiques", ascending=False)
)

display(stat_summary.head(20))


## 14. Contrôle spatial facultatif

Ce contrôle compare, à une résolution horaire, une station avec les stations voisines mesurant la même variable. Il est désactivé par défaut parce qu’il exige plusieurs stations proches, des coordonnées valides et une période commune.

Les stations voisines ne sont pas traitées comme une vérité absolue : le test signale seulement une incohérence locale possible.

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    radius = 6371.0088
    lat1, lon1, lat2, lon2 = map(
        np.radians, [lat1, lon1, lat2, lon2]
    )
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = (
        np.sin(dlat / 2.0) ** 2
        + np.cos(lat1) * np.cos(lat2)
        * np.sin(dlon / 2.0) ** 2
    )
    return 2 * radius * np.arcsin(np.sqrt(a))


qc["flag_spatial_inconsistency"] = False
qc["spatial_neighbor_count"] = 0
qc["spatial_peer_median"] = np.nan
qc["spatial_abs_deviation"] = np.nan

if RUN_SPATIAL_QC:
    spatial_source = qc.dropna(
        subset=["latitude", "longitude", "timestamp", "value_std"]
    ).copy()

    station_coords = (
        spatial_source.groupby("station_id")[["latitude", "longitude"]]
        .median()
        .reset_index()
    )

    hourly = (
        spatial_source.set_index("timestamp")
        .groupby(["station_id", "variable"])
        .resample("1h")["value_std"]
        .median()
        .dropna()
        .reset_index()
        .merge(station_coords, on="station_id", how="left")
    )

    spatial_records = []

    for (variable, timestamp), block in hourly.groupby(
        ["variable", "timestamp"]
    ):
        if len(block) < SPATIAL_MIN_NEIGHBORS + 1:
            continue

        for _, row in block.iterrows():
            distances = haversine_km(
                row["latitude"],
                row["longitude"],
                block["latitude"].to_numpy(),
                block["longitude"].to_numpy(),
            )

            neighbor_mask = (
                (block["station_id"].to_numpy() != row["station_id"])
                & (distances <= SPATIAL_RADIUS_KM)
            )
            neighbors = block.loc[
                neighbor_mask, "value_std"
            ].dropna()

            if len(neighbors) < SPATIAL_MIN_NEIGHBORS:
                continue

            peer_median = neighbors.median()
            peer_mad = np.median(
                np.abs(neighbors - peer_median)
            )
            abs_deviation = abs(
                row["value_std"] - peer_median
            )

            dynamic_threshold = (
                SPATIAL_ROBUST_Z_THRESHOLD * 1.4826 * peer_mad
                if peer_mad > 0 else 0.0
            )
            threshold = max(
                SPATIAL_ABS_FLOOR.get(variable, 0.0),
                dynamic_threshold,
            )

            spatial_records.append({
                "station_id": row["station_id"],
                "variable": variable,
                "timestamp_hour": timestamp,
                "spatial_neighbor_count": len(neighbors),
                "spatial_peer_median": peer_median,
                "spatial_abs_deviation": abs_deviation,
                "flag_spatial_inconsistency": (
                    abs_deviation > threshold
                ),
            })

    spatial_hourly = pd.DataFrame(spatial_records)

    if not spatial_hourly.empty:
        qc["timestamp_hour"] = qc["timestamp"].dt.floor("1h")
        qc = qc.merge(
            spatial_hourly,
            on=["station_id", "variable", "timestamp_hour"],
            how="left",
            suffixes=("", "_new"),
        )

        for col in [
            "spatial_neighbor_count",
            "spatial_peer_median",
            "spatial_abs_deviation",
            "flag_spatial_inconsistency",
        ]:
            new_col = f"{col}_new"
            if new_col in qc.columns:
                qc[col] = qc[new_col].combine_first(qc[col])
                qc.drop(columns=new_col, inplace=True)

        qc["flag_spatial_inconsistency"] = (
            qc["flag_spatial_inconsistency"]
            .fillna(False)
            .astype(bool)
        )
        qc["spatial_neighbor_count"] = (
            qc["spatial_neighbor_count"]
            .fillna(0)
            .astype(int)
        )
        print("Contrôle spatial exécuté.")
    else:
        print("Contrôle spatial non calculable : voisinage insuffisant.")
else:
    print("Contrôle spatial désactivé.")

## 15. Synthèse des drapeaux de qualité

In [ ]:
FLAG_COLUMNS = [
    "flag_unit_unknown",
    "flag_duplicate_timestamp",
    "flag_physical_implausibility",
    "flag_gap_before",
    "flag_stuck_value",
    "flag_temporal_jump",
    "flag_statistical_anomaly",
    "flag_spatial_inconsistency",
]

for col in FLAG_COLUMNS:
    if col not in qc.columns:
        qc[col] = False
    qc[col] = qc[col].fillna(False).astype(bool)

qc["qc_flag_count"] = qc[FLAG_COLUMNS].sum(axis=1)

major_flags = [
    "flag_duplicate_timestamp",
    "flag_physical_implausibility",
    "flag_stuck_value",
    "flag_temporal_jump",
    "flag_statistical_anomaly",
    "flag_spatial_inconsistency",
]

review_flags = [
    "flag_unit_unknown",
    "flag_gap_before",
]

qc["qc_status"] = "valide"
qc.loc[
    qc[review_flags].any(axis=1),
    "qc_status"
] = "a_verifier"
qc.loc[
    qc[major_flags].any(axis=1),
    "qc_status"
] = "suspecte"

status_summary = (
    qc["qc_status"]
    .value_counts(dropna=False)
    .rename_axis("statut")
    .reset_index(name="nombre")
)
status_summary["proportion"] = (
    status_summary["nombre"] / len(qc)
)

display(status_summary)

## 16. Métriques par station et variable

In [ ]:
def proportion(series):
    return float(series.mean()) if len(series) else np.nan


series_metrics = (
    qc.groupby(SERIES_COLS, dropna=False)
    .agg(
        observations_qc=("value_std", "size"),
        station_name=("station_name", "first"),
        latitude=("latitude", "median"),
        longitude=("longitude", "median"),
        unit_std=("unit_std", "first"),
        unit_unknown_rate=("flag_unit_unknown", proportion),
        duplicate_rate=("flag_duplicate_timestamp", proportion),
        physical_anomaly_rate=("flag_physical_implausibility", proportion),
        gap_event_rate=("flag_gap_before", proportion),
        stuck_value_rate=("flag_stuck_value", proportion),
        temporal_jump_rate=("flag_temporal_jump", proportion),
        statistical_anomaly_rate=("flag_statistical_anomaly", proportion),
        spatial_inconsistency_rate=("flag_spatial_inconsistency", proportion),
        valid_observation_rate=("qc_status", lambda s: (s == "valide").mean()),
        suspect_observation_rate=("qc_status", lambda s: (s == "suspecte").mean()),
    )
    .reset_index()
)

series_metrics = series_metrics.merge(
    series_coverage,
    on=SERIES_COLS,
    how="left",
)

series_metrics["expected_interval_minutes"] = (
    series_metrics["expected_interval_seconds"] / 60.0
)
series_metrics["completeness_percent"] = (
    100.0 * series_metrics["completeness_rate"]
)

ordered_columns = [
    "station_id", "station_name", "variable", "_series_key",
    "latitude", "longitude", "unit_std",
    "start_time", "end_time",
    "observations_qc", "observed_count", "expected_count",
    "estimated_missing_count", "expected_interval_minutes",
    "completeness_rate", "completeness_percent",
    "longest_gap_hours",
    "unit_unknown_rate", "duplicate_rate",
    "physical_anomaly_rate", "gap_event_rate",
    "stuck_value_rate", "temporal_jump_rate",
    "statistical_anomaly_rate", "spatial_inconsistency_rate",
    "valid_observation_rate", "suspect_observation_rate",
]

series_metrics = series_metrics[
    [col for col in ordered_columns if col in series_metrics.columns]
]

display(
    series_metrics.sort_values(
        ["completeness_rate", "suspect_observation_rate"],
        ascending=[True, False],
    ).head(30)
)

## 17. Visualisations de diagnostic

In [ ]:
plot_data = (
    series_metrics
    .sort_values("completeness_percent")
    .head(30)
    .copy()
)

if not plot_data.empty:
    plot_data["label"] = (
        plot_data["station_id"].astype(str)
        + " — "
        + plot_data["variable"].astype(str)
    )

    fig, ax = plt.subplots(
        figsize=(10, max(4, len(plot_data) * 0.28))
    )
    ax.barh(
        plot_data["label"],
        plot_data["completeness_percent"]
    )
    ax.set_xlabel("Complétude estimée (%)")
    ax.set_ylabel("Station — variable")
    ax.set_title(
        "Séries présentant les plus faibles taux de complétude"
    )
    ax.set_xlim(0, 100)
    plt.tight_layout()
    plt.show()
else:
    print("Aucune série disponible pour la visualisation.")

In [ ]:
if not qc.empty:
    example_key = (
        series_metrics
        .sort_values(
            "suspect_observation_rate",
            ascending=False
        )
        .iloc[0]["_series_key"]
    )

    example_variable = (
        series_metrics.loc[
            series_metrics["_series_key"].eq(example_key),
            "variable"
        ].iloc[0]
    )

    example = qc[
        qc["_series_key"].eq(example_key)
        & qc["variable"].eq(example_variable)
    ].sort_values("timestamp")

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(
        example["timestamp"],
        example["value_std"],
        marker=".",
        linestyle="-",
        label="Observations",
    )

    suspicious = example[
        example["qc_status"].eq("suspecte")
    ]
    if not suspicious.empty:
        ax.scatter(
            suspicious["timestamp"],
            suspicious["value_std"],
            marker="x",
            s=50,
            label="Observations suspectes",
        )

    unit_label = (
        example["unit_std"].dropna().iloc[0]
        if example["unit_std"].notna().any()
        else ""
    )

    ax.set_title(
        f"Série contrôlée — {example_key} — {example_variable}"
    )
    ax.set_xlabel("Temps")
    ax.set_ylabel(f"Valeur ({unit_label})")
    ax.legend()
    plt.tight_layout()
    plt.show()

## 18. Export des résultats

In [ ]:
qc_export = qc.drop(
    columns=["_series_key"],
    errors="ignore"
).copy()

metrics_export = series_metrics.rename(
    columns={"_series_key": "series_key"}
).copy()

qc_csv_path = QC_FLAGGED_CSV
metrics_csv_path = QC_METRICS_CSV
rejected_csv_path = QC_REJECTED_CSV
duplicates_csv_path = QC_DUPLICATES_CSV
parameters_path = QC_PARAMS_JSON

qc_export.to_csv(qc_csv_path, index=False)
metrics_export.to_csv(metrics_csv_path, index=False)
rejected_rows.to_csv(rejected_csv_path, index=False)
duplicates_removed.to_csv(duplicates_csv_path, index=False)

parameters = {
    "target_variables": TARGET_VARIABLES,
    "plausibility_limits": PLAUSIBILITY_LIMITS,
    "canonical_units": CANONICAL_UNITS,
    "gap_factor": GAP_FACTOR,
    "stuck_min_duration_hours": STUCK_MIN_DURATION_HOURS,
    "stuck_min_observations": STUCK_MIN_OBSERVATIONS,
    "rounding_digits": ROUNDING_DIGITS,
    "jump_mad_factor": JUMP_MAD_FACTOR,
    "min_jump_rate_per_hour": MIN_JUMP_RATE_PER_HOUR,
    "robust_z_threshold": ROBUST_Z_THRESHOLD,
    "min_group_size_for_stat_qc": MIN_GROUP_SIZE_FOR_STAT_QC,
    "allow_unit_inference": ALLOW_UNIT_INFERENCE,
    "run_spatial_qc": RUN_SPATIAL_QC,
    "spatial_radius_km": SPATIAL_RADIUS_KM,
    "spatial_min_neighbors": SPATIAL_MIN_NEIGHBORS,
    "spatial_robust_z_threshold": SPATIAL_ROBUST_Z_THRESHOLD,
    "spatial_abs_floor": SPATIAL_ABS_FLOOR,
    "aligned_with": "implement_trust_framework.py (partie QC)",
    "notes": "Preuves de qualité, non certification métrologique.",
}

with open(parameters_path, "w", encoding="utf-8") as file:
    json.dump(parameters, file, ensure_ascii=False, indent=2)

parquet_path = PROCESSED_DIR / "opensensemap_measurements_qc_flagged.parquet"
try:
    qc_export.to_parquet(parquet_path, index=False)
    parquet_message = str(parquet_path)
except Exception as exc:
    parquet_message = f"Non produit ({type(exc).__name__}: {exc})"

print("Exports réalisés :")
print("-", qc_csv_path)
print("-", metrics_csv_path)
print("-", rejected_csv_path)
print("-", duplicates_csv_path)
print("-", parameters_path)
print("-", parquet_message)
print(f"\nObservations QC : {len(qc_export):,} | séries métriques : {len(metrics_export):,}")
print("Étape suivante → Notebook 3 (score de confiance).")


## 19. Bilan du Notebook 2

À l’issue de ce notebook, chaque observation retenue dispose de :

- sa valeur originale et sa valeur standardisée ;
- son unité d’origine et son unité canonique ;
- des drapeaux de qualité explicites ;
- un statut `valide`, `a_verifier` ou `suspecte`.

Chaque série station–variable dispose de métriques (complétude, gaps, anomalies physiques/statistiques, valeurs figées, sauts, etc.).

### Étape suivante

Exécuter le **Notebook 3** (`Notebook 3  Score de confiance et classification fit for purpose.ipynb`) pour construire le score de confiance et les classes *fit for purpose*.

Ne pas confondre : **contrôle qualité ≠ certification métrologique ≠ score de confiance**.
